A SEGUIR ESTÃO OS CÓDIGOS PARA A ESTATÍSTICA INFERÊNCIAL DAS POPULAÇÕES

Inicialização das variáveis que armazenam as populações

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import scikit_posthocs as sp
from statsmodels.stats.proportion import proportions_ztest
from itertools import combinations
from scipy.stats import kruskal
from scipy.stats import f_oneway, chi2_contingency

baep = pd.read_excel('BANCOS/BAEPENDI_PADRAO.xlsx')
elsa = pd.read_excel('BANCOS/ELSA_PADRAO.xlsx')
indios = pd.read_excel('BANCOS/INDIOS_PADRAO.xlsx')
mon = pd.read_excel('BANCOS/MONICA_PADRAO.xlsx')

baep['POPULACAO'] = 'BAEPENDI'
elsa['POPULACAO'] = 'ELSA'
indios['POPULACAO'] = 'INDIOS'
mon['POPULACAO'] = 'MONICA'

df = pd.concat([baep, elsa, indios, mon], ignore_index=True)

numericas = ['IDADE', 'VOP', 'PAM', 'PAS', 'PAD']





Esta célula lê quatro bases de dados distintas, identifica a população de origem de cada registro e as unifica em um único DataFrame para permitir comparações entre grupos. Em seguida, as variáveis são separadas em numéricas e categóricas. As variáveis numéricas são comparadas entre as populações por meio da ANOVA de uma via, após a remoção de valores ausentes e verificação de dados suficientes em cada grupo, sendo também calculadas as médias por população. As variáveis categóricas são analisadas utilizando o teste do Qui-quadrado de independência, a partir de tabelas de contingência entre a variável e a população, com cálculo das proporções por grupo e verificação de frequências esperadas baixas. Por fim, os resultados estatísticos, p-valores e medidas descritivas são organizados em um único DataFrame para visualização e eventual exportação.


In [ ]:
# Numéricas → ANOVA
# Categóricas → Qui-quadrado

df = pd.concat([baep, elsa, indios, mon], ignore_index=True)

numericas = ['IDADE', 'VOP', 'PAM', 'PAS', 'PAD']
categoricas = ['SEXO', 'HAS', 'DM', 'TABAGISMO', 'DILISPIDEMIA', 'FATOR_RISCO', 'ETNIA', 'OBESIDADE']

resultados = []

for var in numericas:
    grupos = [g[var].dropna() for _, g in df.groupby('POPULACAO')]
    if all(len(g) > 1 for g in grupos):
        stat, p = f_oneway(*grupos)
        medias = df.groupby('POPULACAO')[var].mean().to_dict()
        resultados.append({
            'Variável': var,
            'Tipo': 'Numérica',
            'Estatística_F': stat,
            'p-valor': p,
            'Significativo (p<0.05)': p < 0.05,
            'Medias_Populacao': medias
        })
    else:
        resultados.append({
            'Variável': var,
            'Tipo': 'Numérica',
            'Estatística_F': None,
            'p-valor': None,
            'Significativo (p<0.05)': None,
            'Medias_Populacao': None
        })

for var in categoricas:
    tabela = pd.crosstab(df[var], df['POPULACAO'])
    if tabela.shape[0] > 1 and tabela.shape[1] > 1:
        chi2, p, dof, expected = chi2_contingency(tabela)
        if (expected < 5).any():
            print(f"Atenção: Frequência esperada <5 para variável {var}. p-valor pode não ser confiável.")
        proporcoes = tabela.div(tabela.sum(axis=0), axis=1).to_dict()
        resultados.append({
            'Variável': var,
            'Tipo': 'Categórica',
            'Estatística_F': chi2,
            'p-valor': p,
            'Significativo (p<0.05)': p < 0.05,
            'Proporcoes_Populacao': proporcoes
        })
    else:
        resultados.append({
            'Variável': var,
            'Tipo': 'Categórica',
            'Estatística_F': None,
            'p-valor': None,
            'Significativo (p<0.05)': None,
            'Proporcoes_Populacao': None
        })

resultados_df = pd.DataFrame(resultados)

pd.set_option('display.max_columns', None)
print(resultados_df)

# resultados_df.to_excel('p_valores_anova_quadrado.xlsx', index=False)

Aqui realiza-se comparações estatísticas par a par entre as populações para variáveis categóricas binárias. Para cada variável, são formados todos os pares possíveis de populações e, para cada par, calcula-se o número de ocorrências da categoria de interesse e o total de observações em cada grupo. Em seguida, é aplicado o teste z para comparação de proporções, que avalia se a proporção da variável difere entre duas populações específicas. O p-valor de cada comparação é exibido no console, permitindo identificar diferenças estatisticamente significativas entre pares de populações.

In [ ]:
# --

categoricas = ['SEXO', 'HAS', 'DM', 'TABAGISMO', 'DILISPIDEMIA', 'FATOR_RISCO', 'ETNIA', 'OBESIDADE']

for var in categoricas:
    print(f"\nComparações par a par para {var}:")
    pop_groups = df['POPULACAO'].unique()
    for g1, g2 in combinations(pop_groups, 2):
        contagem = df[df['POPULACAO'].isin([g1, g2])].groupby('POPULACAO')[var].sum().values
        nobs = df[df['POPULACAO'].isin([g1, g2])].groupby('POPULACAO')[var].count().values
        stat, p = proportions_ztest(count=contagem, nobs=nobs)
        print(f"{g1} vs {g2} -> p-valor: {p:.4f}")

# --

Esta célula realiza comparações par a par entre as populações para variáveis numéricas utilizando o teste post-hoc de Dunn. Para cada variável numérica, os dados são filtrados para remover valores ausentes e agrupados por população. O teste de Dunn é então aplicado para avaliar diferenças entre todos os pares de populações, com ajuste de Bonferroni para controle do erro do tipo I devido às múltiplas comparações. Os p-valores ajustados de cada comparação são exibidos, permitindo identificar quais pares de populações apresentam diferenças estatisticamente significativas.

In [ ]:
numericas = ['IDADE', 'VOP', 'PAM', 'PAS', 'PAD']

for var in numericas:
    data = df[[var, 'POPULACAO']].dropna()
    dunn = sp.posthoc_dunn(data, val_col=var, group_col='POPULACAO', p_adjust='bonferroni')
    print(f"\nPost-hoc Dunn para {var} (p-valor ajustado):")
    print(dunn)

O código realiza comparações estatísticas par a par entre as populações para variáveis categóricas, selecionando automaticamente o teste mais apropriado de acordo com a estrutura dos dados. Para cada variável, são considerados apenas os registros válidos e formados todos os pares possíveis de populações. Para cada par, é construída uma tabela de contingência e, quando há frequências pequenas, é aplicado o teste exato de Fisher nos casos 2×2; caso contrário, utiliza-se o teste do Qui-quadrado de independência. Os p-valores obtidos em cada comparação são então corrigidos pelo método de Bonferroni para controle do erro devido a múltiplas comparações. Por fim, os resultados corrigidos são organizados e exibidos, indicando quais comparações apresentam diferença estatisticamente significativa.

In [ ]:
import pandas as pd
import itertools
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests

categoricas = ['SEXO', 'HAS', 'DM', 'TABAGISMO', 'DILISPIDEMIA', 'FATOR_RISCO', 'ETNIA', 'OBESIDADE']

populacoes = df['POPULACAO'].unique()

for var in categoricas:
    print(f"\n=== Comparações para {var} ===")

    data = df[[var, 'POPULACAO']].dropna()

    results = []

    for pop1, pop2 in itertools.combinations(populacoes, 2):
        subset = data[data['POPULACAO'].isin([pop1, pop2])]

        tabela = pd.crosstab(subset['POPULACAO'], subset[var])

        if (tabela.values < 5).any():
            if tabela.shape == (2, 2):
                _, p = fisher_exact(tabela)
            else:
                _, p, _, _ = chi2_contingency(tabela)
        else:
            _, p, _, _ = chi2_contingency(tabela)

        results.append({
            'Comparação': f"{pop1} vs {pop2}",
            'p-valor bruto': p
        })

    pvals = [r['p-valor bruto'] for r in results]
    rejeita, pvals_corr, _, _ = multipletests(pvals, method='bonferroni')

    for i, r in enumerate(results):
        r['p-ajustado (Bonferroni)'] = pvals_corr[i]
        r['Significativo'] = 'Sim' if rejeita[i] else 'Não'

    resultados_df = pd.DataFrame(results)
    print(resultados_df.to_string(index=False))
